In [ ]:
# import modules
import numpy as np
import tensorflow as tf
from tensorflow import keras
import pandas as pd
import seaborn as sns
from pylab import rcParams
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupShuffleSplit
from scipy import stats
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from tensorflow.keras import optimizers
from datetime import datetime
from kerastuner.tuners import RandomSearch
import tensorflow_addons as tfa
from keras.utils import to_categorical 
from keras.regularizers import l2
from scipy.stats import mode



r_seed = 42

WORKDIR = 'C:/Users/cristiano.nascimento/git/LSTM_giroscopio_acelerometro'
np.random.seed(r_seed)
tf.random.set_seed(r_seed)
data_hora= datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
LOG_DIR = WORKDIR + "/Code/DataPrep/"+data_hora

#pd.set_option('display.float_format', lambda x: f'{x:.17f}')

In [ ]:
date = datetime.now()
date = date.strftime("%d/%m/%Y_%H:%M")
arq = datetime.now()
arq = arq.strftime("%d_%m_%Y_%H_%M")

In [ ]:
## get dataset
pd.set_option("display.precision", 9)

# select only sensor data from smart watches
adl_data = np.load(WORKDIR+'/Data/UNIMIB-SHAR/adl_data.npy',allow_pickle=True)

In [ ]:
adl_data = pd.DataFrame(adl_data)

In [ ]:
adl_data.head()

In [ ]:
adl_data.shape

In [ ]:
adl_labels = np.load(WORKDIR+'/Data/UNIMIB-SHAR/adl_labels.npy')

In [ ]:
adl_labels.shape

In [ ]:
adl_labels = pd.DataFrame(adl_labels)

In [ ]:
adl_labels.head()

In [ ]:
adl_labels[0].value_counts().sort_index()

In [ ]:
adl_data[453] = adl_labels[1]

In [ ]:
adl_data[454] = adl_labels[0]

In [ ]:
adl_data.tail()

In [ ]:
adl_data[454].value_counts().sort_index()

In [ ]:
adl_data[453].value_counts().sort_index()

In [ ]:
import pandas as pd

def transform_accelerometer_data(df):
    """
    Transforms a dataframe of accelerometer data where each row contains a 1x453 vector
    of accelerometer readings (x, y, z), a subject_id, and a label into a long format dataframe.

    Args:
        df (pd.DataFrame): Input dataframe with shape (n, 455).
                           Columns 0-452: Accelerometer data (x, y, z groups),
                           Column 453: Subject ID,
                           Column 454: Label ID.

    Returns:
        pd.DataFrame: Transformed dataframe with columns ['x', 'y', 'z', 'subject_id', 'label'].
    """
    # Initialize an empty list to store transformed data
    transformed_data = []

    # Iterate through each row in the dataframe
    for index, row in df.iterrows():
        # Extract subject_id and label
        subject_id = row[453]
        label = row[454]
        
        # Iterate through the accelerometer data in steps of 3 (x, y, z)
        for i in range(0, 453, 3):
            x = row[i]
            y = row[i + 1]
            z = row[i + 2]
            
            # Append the new row to the transformed_data list
            transformed_data.append([x, y, z, subject_id, label])

    # Convert the transformed data into a new dataframe
    transformed_df = pd.DataFrame(transformed_data, columns=['x', 'y', 'z', 'subject_id', 'label'])

    return transformed_df


In [ ]:
import pandas as pd

# Example function to organize data by blocks of 'atividade'
def organize_blocks_by_atividade(data):
    """
    Organizes the dataframe by extracting blocks of the same 'atividade' for each 'pessoa_x'.
    Blocks are concatenated vertically for each 'atividade' value in descending order.

    Args:
        data (pd.DataFrame): Input dataframe with columns including 'pessoa_x', 'atividade', and others.

    Returns:
        pd.DataFrame: New dataframe organized by 'atividade' blocks.
    """
    organized_data = pd.DataFrame(columns=data.columns)  # Create an empty DataFrame with the same columns

    # Loop through each pessoa_x
    for pessoa_id, pessoa_data in data.groupby('label'):
        # Loop through atividade values from 5 to 0
        for atividade_val in range(10, -1, -1):
            # Get blocks of data for the current 'atividade'
            atividade_block = pessoa_data[pessoa_data['label'] == atividade_val]
            # Append to the organized_data
            organized_data = pd.concat([organized_data, atividade_block], axis=0)

    # Reset the index to maintain sequential indexing in the new dataframe
    organized_data.reset_index(drop=True, inplace=True)

    return organized_data

# Assuming 'data' is already loaded as a DataFrame
# organized_data = organize_blocks_by_atividade(data)



In [ ]:
import pandas as pd
import numpy as np

def noise_reduction(df, window_size=5):
    """
    Apply noise reduction using a moving average filter to accelerometer and gyroscope data.

    Parameters:
    df (pd.DataFrame): Input dataframe of shape (n, 10) with the first 6 columns containing raw accelerometer and gyroscope data.
    window_size (int): The size of the moving window for the moving average filter. Default is 5.

    Returns:
    pd.DataFrame: DataFrame with noise-reduced accelerometer and gyroscope data.
    """
    # Ensure the dataframe has at least 6 columns
    if df.shape[1] < 3:
        raise ValueError("Input dataframe must have at least 6 columns.")
    
    # Extract the first 6 columns (accelerometer and gyroscope data)
    acc_gyro_data = df.iloc[:, :3]
    
    # Apply a moving average filter to reduce noise
    acc_gyro_smoothed = acc_gyro_data.rolling(window=window_size, min_periods=1, center=True).mean()
    
    # Replace the first 6 columns with the smoothed data
    df.iloc[:, :3] = acc_gyro_smoothed
    
    return df




In [ ]:
transformed_adl_data = transform_accelerometer_data(adl_data)

In [ ]:
transformed_adl_data.head()

In [ ]:
transformed_adl_data = noise_reduction(transformed_adl_data)

In [ ]:
transformed_adl_data.shape

In [ ]:
transformed_adl_data = organize_blocks_by_atividade(transformed_adl_data)

In [ ]:
transformed_adl_data.subject_id = transformed_adl_data.subject_id.astype(int)

In [ ]:
transformed_adl_data.subject_id.value_counts().sort_index()

In [ ]:
transformed_adl_data.shape

In [ ]:
transformed_adl_data = transformed_adl_data.dropna()

In [ ]:
transformed_adl_data.shape

In [ ]:
print(transformed_adl_data['x'].iloc[0]) 

In [ ]:
transformed_adl_data.label.value_counts().sort_index()

In [ ]:
""" import pandas as pd

# Load the labels.txt file into a DataFrame
labels = pd.read_csv(WORKDIR+'labels.txt', sep=' ', header=None, 
                     names=['experiment', 'user', 'activity', 'start', 'end'])

# Create a 'label' column in data_df with default NaN values
data_df['label'] = pd.NA

# Process each row in the labels DataFrame
for _, row in labels.iterrows():
    # Extract the interval and corresponding user and experiment
    start, end = row['start'], row['end']
    user, experiment = row['user'], row['experiment']
    activity = row['activity']
    
    # Find the index interval in data_df
    interval_mask = (data_df.index >= start) & (data_df.index <= end)
    
    # Verify if user and experiment match
    if all(data_df.loc[interval_mask, 'user'] == user) and \
       all(data_df.loc[interval_mask, 'experiment'] == experiment):
        # Assign the activity label to the 'label' column in the specified interval
        data_df.loc[interval_mask, 'label'] = activity
 """

In [ ]:
""" import pandas as pd
import numpy as np

def label_sensor_data(data_df, labels_path):
   
    # Create a copy of the dataframe to avoid modifying the original
    df = data_df.copy()
    
    # Initialize the 'label' column with a default value (e.g., -1 or 'unknown')
    df['label'] = -1
    
    # Read the labels file
    try:
        labels_df = pd.read_csv(labels_path, header=None, delim_whitespace=True, 
                                names=['experiment_id', 'user_id', 'activity_id', 'start_point', 'end_point'])
    except Exception as e:
        print(f"Error reading labels file: {e}")
        return df
    
    # Process each line in the labels file
    for _, label_row in labels_df.iterrows():
        # Extract label information
        exp_id = label_row['experiment_id']
        user_id = label_row['user_id']
        activity_id = label_row['activity_id']
        start_point = label_row['start_point']
        end_point = label_row['end_point']
        
        # Find the index interval in result_df
        mask = (
            (df['experiment'] == exp_id) & 
            (df['user'] == user_id) & 
            (df.index >= start_point) & 
            (df.index <= end_point)
        )
        
        # Label the rows in the specified interval
        df.loc[mask, 'label'] = activity_id
    
    return df

 """

In [ ]:
""" import pandas as pd

# Read the data_df dataframe (assuming it's already loaded)
# data_df = pd.read_csv('your_file.csv')  # Uncomment this line to load your dataframe

# Read labels.txt file (the labels file should be in the same format as described)
labels_df = pd.read_csv(WORKDIR+'labels.txt', header=None, sep="\s+", names=["experiment", "user", "activity", "start_point", "end_point"])

# Initialize the 'label' column in data_df with NaN (or 0 if you prefer a placeholder)
data_df['label'] = None

# Iterate over the rows in the labels dataframe and map the activities to the data_df
for _, row in labels_df.iterrows():
    experiment_id = row['experiment']
    user_id = row['user']
    activity_id = row['activity']
    start_point = row['start_point']
    end_point = row['end_point']
    
    # Filter the rows in data_df that belong to the same experiment and user
    mask = (data_df['experiment'] == experiment_id) & (data_df['user'] == user_id)
    
    # Update the 'label' column for the corresponding rows based on the sample points
    data_df.loc[mask & (data_df.index >= start_point) & (data_df.index <= end_point), 'label'] = activity_id

# The 'label' column is now populated with the activity labels corresponding to each row
 """

In [ ]:
#selecting only activities 0 to 3
transformed_adl_data = transformed_adl_data[transformed_adl_data['label'].isin([1,2,3,8,9])]

In [ ]:
# Define the mapping dictionary
mapping_dict = {1:0,2:1,3:2,8:3,9:4}

In [ ]:
transformed_adl_data["label"] = transformed_adl_data["label"].replace(to_replace=mapping_dict)

In [ ]:
transformed_adl_data.label.value_counts().sort_index()

In [ ]:
transformed_adl_data.label = transformed_adl_data.label.astype(int)

In [ ]:
transformed_adl_data.tail()

In [ ]:
transformed_adl_data_validation = transformed_adl_data[transformed_adl_data['subject_id'].isin([1,6,11,16,21,22,26,30])]

In [ ]:
transformed_adl_data_validation.head()

In [ ]:
transformed_adl_data_validation.shape

In [ ]:
transformed_adl_data_validation.label.value_counts()

In [ ]:
transformed_adl_data_validation.subject_id.value_counts().sort_index()

In [ ]:
transformed_adl_data_validation = transformed_adl_data_validation.reset_index(drop = True)

In [ ]:
transformed_adl_data_train = transformed_adl_data[~transformed_adl_data['subject_id'].isin([1,6,11,16,21,22,26,30])]


In [ ]:
transformed_adl_data_train.shape

In [ ]:
transformed_adl_data_train.head()

In [ ]:
transformed_adl_data_train.subject_id.value_counts().sort_index()

In [ ]:
transformed_adl_data_train.label.value_counts().sort_index()

In [ ]:
transformed_adl_data_train = transformed_adl_data_train.reset_index(drop = True)

In [ ]:
transformed_adl_data_train.shape

In [ ]:
transformed_adl_data_train.subject_id.value_counts().sort_index()

In [ ]:
grouped_data_organized = transformed_adl_data_train.groupby(['subject_id'])['label'].value_counts().unstack()

In [ ]:
print(grouped_data_organized)

In [ ]:
print(f"percentage of validation is {transformed_adl_data_validation.shape[0]/transformed_adl_data.shape[0]}")

In [ ]:
print(f"percentage of validation is {transformed_adl_data_train.shape[0]/transformed_adl_data.shape[0]}")

# **Models**

In [ ]:
def split_and_shuffle_3d_arrays(array_data, labels_data):
    
    combined = list(zip(array_data, labels_data))


    # Step 2: Separate into four parts based on label value
    # Filtering combined list by labels in each window
    grouped_data = {
        i: [item for item in combined if np.unique(item[1][:, 0])[0] == i] for i in range(3)
    }

    # Step 3: Interlace half of label=1 into half of label=0
    first_third_0 = len(grouped_data[0]) // 3
    half_1 = len(grouped_data[1]) // 3
    interlaced_0 = []

    for i in range(half_0):
        interlaced_0.append(grouped_data[0][i])
        if i < half_1:
            interlaced_0.append(grouped_data[1][i])

    # Step 4: Interlace half of label=3 into half of label=2
    half_2 = len(grouped_data[2]) // 2
    half_3 = len(grouped_data[3]) // 2
    interlaced_2 = []

    for i in range(half_2):
        interlaced_2.append(grouped_data[2][i])
        if i < half_3:
            interlaced_2.append(grouped_data[3][i])

    # Step 5: Interlace last half of label=0 into last half of label=3
    last_half_1 = grouped_data[1][half_1:]
    last_half_0 = grouped_data[0][half_0:]
    interlaced_1 = []

    for i in range(len(last_half_1)):
        interlaced_1.append(last_half_1[i])
        if i < len(last_half_0):
            interlaced_1.append(last_half_0[i])

    # Step 6: Concatenate the three resulting arrays vertically
    combined_result = interlaced_0 + interlaced_2 + interlaced_3

    # Step 7: Unzip the resulting array into separate data and labels
    data_result, labels_result = zip(*combined_result)

    # Converting back to numpy arrays for further processing if needed
    data_result = np.array(data_result)
    labels_result = np.array(labels_result)
    
    return data_result,labels_result



In [ ]:
import numpy as np

def category_count_categorical(labels_test_categorical):

    # Assuming your data is stored in a variable called `data`
    # Replace this with the actual data if it's available

    # Sum along the rows to get the count for each category
    category_counts_categorical = np.sum(labels_test_categorical, axis=0)

    # Print the counts
    print("Category counts:", category_counts_categorical)


In [ ]:
import numpy as np

# Example array with binary class indications, replace this with your actual array
#array_data = np.random.randint(0, 2, (4027, 7)).astype(float)  # Example data; replace with your own

def count_active_classes(array_data):
    # Sum across each row to get the number of classes present in each row
    active_classes_per_row = np.sum(array_data, axis=1)
    
    # Count unique values in active_classes_per_row to get how many rows have each class count
    unique_counts, count_of_rows = np.unique(active_classes_per_row, return_counts=True)
    
    # Map each unique count of active classes to the number of rows with that count
    count_summary = dict(zip(unique_counts.astype(int), count_of_rows))
    
    return count_summary

# Get the count of active classes for each row
#active_classes_count = count_active_classes(array_data)

# Print the results
""" for active_class_count, row_count in active_classes_count.items():
    print(f"Number of rows with {active_class_count} active classes: {row_count}") """


In [ ]:
""" import numpy as np

# Function to count occurrences of each category
def count_categories(data):
    counts = {
        '[1, 0, 0, 0]': np.sum((data == [1., 0., 0., 0.]).all(axis=1)),
        '[0, 1, 0, 0]': np.sum((data == [0., 1., 0., 0.]).all(axis=1)),
        '[0, 0, 1, 0]': np.sum((data == [0., 0., 1., 0.]).all(axis=1)),
        '[0, 0, 0, 1]': np.sum((data == [0., 0., 0., 1.]).all(axis=1)),
    }
    return counts

# Count categories before removal
print("Category counts before removal:", count_categories(labels_data_categorical))

# Define the desired class count for the first category
desired_class_count = 2151

# Identify indices of the first class (1, 0, 0, 0) in labels_data_categorical
first_class_indices = np.where((labels_data_categorical == [1., 0., 0., 0.]).all(axis=1))[0]

# Randomly select indices to remove to balance the first class
indices_to_remove = np.random.choice(first_class_indices, size=(len(first_class_indices) - desired_class_count), replace=False)

# Remove the selected indices from both labels_data_categorical and segments_data
labels_data_categorical = np.delete(labels_data_categorical, indices_to_remove, axis=0)
segments_data = np.delete(segments_data, indices_to_remove, axis=0)

# Count categories after removal
print("Category counts after removal:", count_categories(labels_data_categorical))

# Print the shape of the new arrays to verify
print("New shape of labels_data_categorical:", labels_data_categorical.shape)
print("New shape of segments_data:", segments_data.shape)
 """

In [ ]:
def delete_segments_not_fully_filled(df_data_windowed,df_label_windowed):
    indexes = np.where(df_label_windowed == -1)[0]
    unique_arr = np.unique(indexes, return_index=True)[0]
    new_df_label_windowed = np.delete(df_label_windowed,unique_arr,axis=0)
    new_df_data_windowed = np.delete(df_data_windowed,unique_arr,axis=0)
    return (new_df_data_windowed,new_df_label_windowed)

In [ ]:
import numpy as np

def normalize_segments_to_mode(df_data_windowed, df_label_windowed):
    # Iterate through each window
    for i in range(df_label_windowed.shape[0]):
        # Get the current window
        window = df_label_windowed[i]
        
        # Replace -1 values with the first value of the window
        first_value = window[0]
        print(first_value)
        window = np.where(window == -1, first_value, window)
        
        # Update the window in the array
        df_label_windowed[i] = window
    
    return df_data_windowed, df_label_windowed


In [ ]:
def apply_segmentation_no_sliding_window_validation(df_data):
# Define window size and step size
    window_size = 151
    step_size = 30

    # Calculate number of windows
    n_windows = int((df_data.shape[0] / window_size))

    # Create segmented data using  window
    segments = np.zeros((n_windows, window_size, df_data.shape[1]-2),dtype=np.float32)
    targets = np.full((n_windows, window_size, 1),-1)
    for i in range(n_windows):
        start_idx = i * window_size
        end_idx = start_idx + window_size
        #populate the 3d array only if the 256 segment has the value in label
         # Label consistency check
        label_consistency = (
            df_data[['label']][start_idx:end_idx].nunique()[0] == 1        
        ) 
        if label_consistency == True:
            segments[i] = df_data[['x', 'y', 'z']][start_idx:end_idx]
            targets[i] = df_data[['label']][start_idx:end_idx]
    return (segments,targets)

In [ ]:
def apply_segmentation_no_sliding_window(df_data):
    import numpy as np

    # Define window size and step size
    window_size = 151
    step_size = 30

    # Define the threshold for sufficient transition labels
    transition_threshold = 0.3  # At least 10% of the window size

    # Calculate number of windows
    n_windows = int((df_data.shape[0] / window_size))

    # Create segmented data using sliding window
    segments = np.zeros((n_windows, window_size, df_data.shape[1] - 2), dtype=np.float32)
    targets = np.full((n_windows, window_size, 1), -1)

    for i in range(n_windows):
        start_idx = i * window_size
        end_idx = start_idx + window_size

        # Get the first row value for label

        # Label consistency check
        label_consistency = (
            df_data[['label']][start_idx:end_idx].nunique()[0] == 1
        )

        if label_consistency:
            # Populate the 3D array for consistent labels
            segments[i] = df_data[
                ['x', 'y', 'z']
            ][start_idx:end_idx].values
            targets[i] = df_data[['label']][start_idx:end_idx].values                 

    return segments, targets


In [ ]:
import numpy as np
import pandas as pd

def apply_segmentation_with_sliding_window(df, window_size=151, step_size=30):
    """
    Segment a DataFrame into windows with specific constraints.
    
    Parameters:
    -----------
    df : pandas.DataFrame
        Input DataFrame with columns 'x', 'y', 'z', and a label column
    window_size : int, optional
        Size of each window (default 151)
    step_size : int, optional
        Step size between windows (default 15)
    
    Returns:
    --------
    tuple: (variable_data_array, label_data_array)
        variable_data_array: numpy array of shape (num_windows, window_size, 3)
        label_data_array: numpy array of shape (num_windows, window_size, 1)
    """
    # Extract variable columns and label column
    var_columns = ['x', 'y', 'z', 'subject_id','label']
    label_column = df.columns[df.columns.tolist().index(var_columns[-1])]
    print(label_column)
    # Initialize lists to store valid windows
    variable_windows = []
    label_windows = []
    
    # Iterate through the DataFrame with sliding window
    for start in range(0, len(df) - window_size + 1, step_size):
        # Extract current window
        window = df.iloc[start:start+window_size]
        
        # Check if all labels in the window are the same
        if window[label_column].nunique() == 1:
            # Extract variable data and label data for this window
            var_data = window[var_columns].values
            label_data = window[label_column].values.reshape(-1, 1)
            
            # Append to lists of windows
            variable_windows.append(var_data)
            label_windows.append(label_data)
    
    # Convert lists to numpy arrays
    if variable_windows:
        variable_data_array = np.array(variable_windows)
        label_data_array = np.array(label_windows)
        
        return variable_data_array, label_data_array
    else:
        return np.array([]), np.array([])

# Example usage:
# Assuming 'df' is your input DataFrame
# var_data, label_data = segment_dataframe(df)
# print(var_data.shape, label_data.shape)

In [ ]:
#print(data.pessoa_x.unique())

In [ ]:
#X_test_data = data[data['pessoa_x'].isin([0,1])]

In [ ]:
#print(X_test_data.shape)

In [ ]:
#X_test_data = X_test_data.reset_index(drop = True)

In [ ]:
#X_test_data.head()

In [ ]:
#X_test_data.tail()

In [ ]:
#X_test_data.pessoa_x.unique()

In [ ]:
#print(f"percentage of validation is {X_test_data.shape[0]/data.shape[0]}")

In [ ]:
#X_test_data['atividade'].value_counts()

In [ ]:
#X_test_data = X_test_data[['acelerômetro_x', 'acelerômetro_Y', 'acelerômetro_Z', 'giroscópio_x', 'giroscópio_Y', 'giroscópio_Z','magnetrômetro_x','magnetrômetro_y','magnetrômetro_z','atividade']]

In [ ]:
#X_train_validation_data = data[~data.pessoa_x.isin([0,1])]

In [ ]:
#print(X_train_validation_data.shape)

In [ ]:
#X_train_validation_data = X_train_validation_data.reset_index(drop = True)

In [ ]:
#X_train_validation_data.head()

In [ ]:
#X_train_validation_data.tail()

In [ ]:
#X_train_validation_data.pessoa_x.unique()

In [ ]:
#print(f"percentage of validation is {X_train_validation_data.shape[0]/data.shape[0]}")

In [ ]:
from imblearn.over_sampling import SMOTE
from collections import Counter
import pandas as pd

def balance_data_with_smote(data, label_column):
    """
    Balances the class distribution in a DataFrame using SMOTE.

    Parameters:
        data (pd.DataFrame): The input DataFrame containing features and labels.
        label_column (str): The name of the column containing class labels.

    Returns:
        pd.DataFrame: A balanced DataFrame with the same structure as the input.
    """
    # Separate features and labels
    X = data.drop(columns=[label_column])
    y = data[label_column]

    # Check class distribution before SMOTE
    print("Before SMOTE:", Counter(y))

    # Apply SMOTE
    smote = SMOTE(sampling_strategy='auto', random_state=42)
    X_smote, y_smote = smote.fit_resample(X, y)

    # Check class distribution after SMOTE
    print("After SMOTE:", Counter(y_smote))

    # Combine features and labels back into a DataFrame
    balanced_data = pd.concat([pd.DataFrame(X_smote, columns=X.columns), 
                               pd.DataFrame(y_smote, columns=[label_column])], axis=1)

    return balanced_data

# Example usage:
# balanced_df = balance_data_with_smote(data, label_column='label')


In [ ]:
#X_train_validation_data['atividade'].value_counts()

In [ ]:
#X_train_validation_data['atividade'].value_counts()

In [ ]:
transformed_adl_data_validation.columns

In [ ]:
#transformed_adl_data_validation = transformed_adl_data_validation[['acc_x', 'acc_y', 'acc_z', 'gyro_x', 'gyro_y', 'gyro_z','label']]

In [ ]:
segments_val,targets_val = apply_segmentation_no_sliding_window_validation(transformed_adl_data_validation)
print(targets_val.shape)
print(segments_val.shape)

In [ ]:
""" import numpy as np

# Example 3D array (replace with your actual array)
n = 10
array = np.random.randint(-5, 5, (n, 256, 1))  # Replace with your actual array

# Check if the last dimension has any value = -1
has_minus_one = np.any(targets_val[:, :, 0] == 5)

if has_minus_one:
    print("The array contains at least one -1 in the last dimension.")
else:
    print("The array does not contain any -1 in the last dimension.")
 """

In [ ]:
import numpy as np
from collections import Counter

# Example initialization of the array (replace this with your actual data)
# targets_train_synthetic_transitions = np.random.randint(0, 5, (n, 200, 1))

def count_int_classes_of_array(targets_train):

    # Flatten the last dimension to make it easier to work with
    flattened_targets = targets_train.reshape(targets_train.shape[0], 151)
    print(flattened_targets)

    # Dictionary to store counts of labels where all 200 values are the same
    label_counts = Counter()

    # Iterate through each window
    for window in flattened_targets:
        # Check if all values in the window are the same
        if np.all(window == window[0]):
            # Increment the count for this label
            label_counts[window[0]] += 1

    # Print the final counts
    print("Counts of unique labels where all values in a window are the same:")
    for label, count in label_counts.items():
        print(f"Label {label}: {count}")


In [ ]:
""" import numpy as np
from collections import Counter

# Assuming `array` is your 3D array with shape (n, 256, 1)
# Create a sample array for demonstration
n = 10  # Number of windows
array = np.random.randint(0, 10, (n, 256, 1))  # Replace this with your actual array

# Flatten the 3D array into a 1D array
flattened_array = targets_val.flatten()

# Count the unique values
unique_values, counts = np.unique(flattened_array, return_counts=True)

# Combine the results into a dictionary
value_counts = dict(zip(unique_values, counts))

# Display the counts
print("Unique values and their counts:")
for value, count in value_counts.items():
    print(f"Value {value}: Count {count}")
 """

In [ ]:
""" import numpy as np
import pandas as pd

# Assume df is the 3D numpy array of shape (n, 100, 1)
# Example: df = np.random.randint(0, 10, size=(10, 256, 1))

def count_unique_values_per_window(df):
    # Ensure the input is 3D
    if df.ndim != 3 or df.shape[2] != 1:
        raise ValueError("Input must be a 3D array of shape (n, 256, 1)")
    
    # Reshape the third dimension for easier processing
    reshaped_df = df.reshape(df.shape[0], df.shape[1])

    # List to store counts for each window
    unique_counts = []

    for window in reshaped_df:
        # Count unique values and their frequencies
        unique, counts = np.unique(window, return_counts=True)
        unique_counts.append(dict(zip(unique, counts)))

    return unique_counts

# Example usage
#df = np.random.randint(0, 10, size=(5, 256, 1))  # 5 windows of 100 rows with random integers 0-9
result = count_unique_values_per_window(targets_val)

# Convert the result to a DataFrame for better visualization
result_df = pd.DataFrame(result).fillna(0).astype(int)

# Display the result
print(result_df)
 """

In [ ]:
segments_val,targets_val = delete_segments_not_fully_filled(segments_val,targets_val)
print(targets_val.shape)
print(segments_val.shape)

In [ ]:
count_active_classes(targets_val)

In [ ]:
count_int_classes_of_array(targets_val)

In [ ]:
#segments_val,targets_val = normalize_segments_to_mode(segments_val,targets_val)

In [ ]:
transformed_adl_data_train.columns

In [ ]:
#transformed_adl_data_train = transformed_adl_data_train[['acc_x', 'acc_y', 'acc_z', 'gyro_x', 'gyro_y', 'gyro_z','label']]

In [ ]:
segments_train,targets_train = apply_segmentation_with_sliding_window(transformed_adl_data_train)
print(targets_train.shape)
print(segments_train.shape)

In [ ]:
segments_train = segments_train[:, :, :-2]



In [ ]:
print(segments_train)

In [ ]:
print(segments_train.shape)

In [ ]:
count_active_classes(targets_train)

In [ ]:
count_int_classes_of_array(targets_train)

In [ ]:
segments_train,targets_train = delete_segments_not_fully_filled(segments_train,targets_train)
print(targets_train.shape)
print(segments_train.shape)

In [ ]:
transformed_adl_data_train.shape

In [ ]:
segments_train_no_sliding_window,targets_train_no_sliding_window = apply_segmentation_no_sliding_window(transformed_adl_data_train)
print(targets_train_no_sliding_window.shape)
print(segments_train_no_sliding_window.shape)

In [ ]:
count_active_classes(targets_train_no_sliding_window)

In [ ]:
count_int_classes_of_array(targets_train_no_sliding_window)

In [ ]:
segments_train_no_sliding_window,targets_train_no_sliding_window = delete_segments_not_fully_filled(segments_train_no_sliding_window,targets_train_no_sliding_window)
print(targets_train_no_sliding_window.shape)
print(segments_train_no_sliding_window.shape)

In [ ]:
print(segments_train_no_sliding_window)

In [ ]:
""" import numpy as np
from collections import Counter

# Assuming `array` is your 3D array with shape (n, 256, 1)
# Create a sample array for demonstration
n = 10  # Number of windows
array = np.random.randint(0, 10, (n, 250, 1))  # Replace this with your actual array

# Flatten the 3D array into a 1D array
flattened_array = targets_val.flatten()

# Count the unique values
unique_values, counts = np.unique(flattened_array, return_counts=True)

# Combine the results into a dictionary
value_counts = dict(zip(unique_values, counts))

# Display the counts
print("Unique values and their counts:")
for value, count in value_counts.items():
    print(f"Value {value}: Count {count}")
 """

In [ ]:
""" import numpy as np
import pandas as pd

# Assume df is the 3D numpy array of shape (n, 256, 1)
# Example: df = np.random.randint(0, 10, size=(10, 256, 1))

def count_unique_values_per_window(df):
    # Ensure the input is 3D
    if df.ndim != 3 or df.shape[2] != 1:
        raise ValueError("Input must be a 3D array of shape (n, 256, 1)")
    
    # Reshape the third dimension for easier processing
    reshaped_df = df.reshape(df.shape[0], df.shape[1])

    # List to store counts for each window
    unique_counts = []

    for window in reshaped_df:
        # Count unique values and their frequencies
        unique, counts = np.unique(window, return_counts=True)
        unique_counts.append(dict(zip(unique, counts)))

    return unique_counts

# Example usage
#df = np.random.randint(0, 10, size=(5, 256, 1))  # 5 windows of 256 rows with random integers 0-9
result = count_unique_values_per_window(targets_val)

# Convert the result to a DataFrame for better visualization
result_df = pd.DataFrame(result).fillna(0).astype(int)

# Display the result
print(result_df)
 """

In [ ]:
""" import numpy as np

# Example 3D array (replace with your actual array)
n = 10
array = np.random.randint(-5, 5, (n, 256, 1))  # Replace with your actual array

# Check if the last dimension has any value = -1
has_minus_one = np.any(targets_val[:, :, 0] == 5)

if has_minus_one:
    print("The array contains at least one -1 in the last dimension.")
else:
    print("The array does not contain any -1 in the last dimension.")
 """

In [ ]:
targets_val = targets_val.reshape(targets_val.shape[0],151)

In [ ]:
print(targets_val.shape)

In [ ]:
count_active_classes(targets_val)

In [ ]:
print(targets_val)

In [ ]:
labels_val = targets_val[:,0]

In [ ]:
print(labels_val)

In [ ]:
print(labels_val.shape)

In [ ]:
labels_val_categorical = to_categorical(labels_val,num_classes=5)

In [ ]:
print(labels_val_categorical)

In [ ]:
print(labels_val_categorical.shape)

In [ ]:
category_count_categorical(labels_val_categorical)

In [ ]:
import numpy as np
from scipy.interpolate import interp1d

def augment_data_with_time_warping(variable_data, label_data, warp_range=(0.95, 1.10)):
    """
    Augments data using Time Warping for windows with unique labels 3, 4, or 5.

    Args:
        variable_data (np.array): 3D array of variable data with shape (n, 100, 7).
        label_data (np.array): 2D array of label data with shape (n, 100, 1).
        warp_range (tuple): Range for time compression/stretching (default is ±5%-10%).

    Returns:
        tuple: Augmented variable data and label data as separate arrays.
    """
    def time_warp(window, warp_factor):
        """
        Apply time warping to a given window.

        Args:
            window (np.array): Input window with shape (100, 7).
            warp_factor (float): Factor to compress/stretch time.

        Returns:
            np.array: Time-warped window with shape (100, 7).
        """
        original_time = np.linspace(0, 1, window.shape[0])
        warped_time = np.linspace(0, 1, max(2, int(window.shape[0] * warp_factor)))  # Ensure at least 2 points

        # Interpolate each feature column separately
        interpolated_window = []
        for i in range(window.shape[1]):  # Iterate over each feature (axis=1)
            interpolator = interp1d(
                warped_time, 
                np.interp(warped_time, original_time, window[:, i]), 
                kind='linear', 
                fill_value="extrapolate", 
                bounds_error=False
            )
            interpolated_window.append(interpolator(original_time))

        return np.stack(interpolated_window, axis=1)

    # Step 1: Zip the two arrays
    zipped_data = list(zip(variable_data, label_data))

    # To hold augmented data
    augmented_variable_data = []
    augmented_label_data = []
    labels_count =  {"label_0": 0, "label_1": 0, "label_2":0, "label_3": 0, "label_4": 0, "label_5": 0, "label_6": 0}

    # Step 2, 3, 4, 5: Iterate through each window and perform operations
    for var_window, label_window in zipped_data:
        # Check if label in window is unique
        unique_labels = np.unique(label_window)
        if len(unique_labels) == 1 and unique_labels[0] != 2:
            label = unique_labels[0]  # Unique label
            #augmented_variable_data.append(var_window)
            #augmented_label_data.append(label_window)

            # Define number of synthetic windows based on label
            
            if label == 1 and labels_count["label_1"] > 1577:
                continue
            elif label == 0 and labels_count["label_0"] > 1632:
                continue
            elif label == 3 and labels_count["label_3"] > 1503:
                continue
            elif label == 4 and labels_count["label_4"] > 1581:
                continue
            
                continue
            
            if label == 1:
                labels_count["label_1"] += 11
                synthetic_count = 11
            elif label == 0:
                labels_count["label_0"] += 16
                synthetic_count = 16
            elif label == 3:
                labels_count["label_3"] += 8
                synthetic_count = 8
            elif label == 4:
                labels_count["label_4"] += 11
                synthetic_count = 11               
            else:
                synthetic_count = 0

            # Generate synthetic windows using Time Warping
            for _ in range(synthetic_count):
                warp_factor = np.random.uniform(*warp_range)
                synthetic_window = time_warp(var_window, warp_factor)
                augmented_variable_data.append(synthetic_window)
                augmented_label_data.append(label_window)  # Use the same label window

    # Step 6: Unzip the augmented arrays
    augmented_variable_data = np.array(augmented_variable_data)
    augmented_label_data = np.array(augmented_label_data)

    return augmented_variable_data, augmented_label_data

# Example usage:
# variable_data = np.random.rand(1000, 100, 7)  # Replace with actual data
# label_data = np.random.randint(0, 6, (1000, 100, 1))  # Replace with actual data
# augmented_var, augmented_labels = augment_data_with_time_warping(variable_data, label_data)
# print("Augmented Variable Data Shape:", augmented_var.shape)
# print("Augmented Label Data Shape:", augmented_labels.shape)


In [ ]:
import numpy as np

def augment_data_with_magnitude_scaling(variable_data, label_data, scale_range=(0.9, 1.1)):
    """
    Augments data using magnitude scaling for windows with unique labels 3, 4, or 5.

    Args:
        variable_data (np.array): 3D array of variable data with shape (n, 100, 7).
        label_data (np.array): 2D array of label data with shape (n, 100, 1).
        scale_range (tuple): Range for scaling magnitude (default is ±10%).

    Returns:
        tuple: Augmented variable data and label data as separate arrays.
    """

    def magnitude_scaling(window, scale_factor):
        """
        Apply magnitude scaling to a given window.
        Args:
            window (np.array): Input window with shape (100, 7).
            scale_factor (float): Factor to scale magnitudes.
        Returns:
            np.array: Scaled window with shape (100, 7).
        """
        return window * scale_factor

    # Step 1: Zip the two arrays
    zipped_data = list(zip(variable_data, label_data))

    # To hold augmented data
    augmented_variable_data = []
    augmented_label_data = []
    labels_count =  {"label_0": 0, "label_1": 0, "label_2":0, "label_3": 0, "label_4": 0, "label_5": 0, "label_6": 0}

    # Step 2, 3, 4, 5: Iterate through each window and perform operations
    for var_window, label_window in zipped_data:
        # Check if label in window is unique
        unique_labels = np.unique(label_window)
        if len(unique_labels) == 1 and unique_labels[0] != 2:
            label = unique_labels[0]  # Unique label
            #augmented_variable_data.append(var_window)
            #augmented_label_data.append(label_window)
            
            # Define number of synthetic windows based on label
            if label == 1 and labels_count["label_1"] > 1577:
                continue
            elif label == 0 and labels_count["label_0"] > 1632:
                continue
            elif label == 3 and labels_count["label_3"] > 1503:
                continue
            elif label == 4 and labels_count["label_4"] > 1581:
                continue
            
                continue
            
            if label == 1:
                labels_count["label_1"] += 11
                synthetic_count = 11
            elif label == 0:
                labels_count["label_0"] += 16
                synthetic_count = 16
            elif label == 3:
                labels_count["label_3"] += 8
                synthetic_count = 8
            elif label == 4:
                labels_count["label_4"] += 11
                synthetic_count = 11               
            else:
                synthetic_count = 0
            
            # Generate synthetic windows using magnitude scaling
            for _ in range(synthetic_count):
                scale_factor = np.random.uniform(*scale_range)
                synthetic_window = magnitude_scaling(var_window, scale_factor)
                augmented_variable_data.append(synthetic_window)
                augmented_label_data.append(label_window)  # Use the same label window

    # Step 6: Unzip the augmented arrays
    augmented_variable_data = np.array(augmented_variable_data)
    augmented_label_data = np.array(augmented_label_data)

    return augmented_variable_data, augmented_label_data


In [ ]:
import numpy as np

def augment_data_with_jittering(variable_data, label_data): 
    def jitter(window, sigma=0.01):
        """
        Adds random Gaussian noise to the input window.
        :param window: The input data window (numpy array).
        :param sigma: Standard deviation of the Gaussian noise.
        :return: Jittered window.
        """
        noise = np.random.normal(loc=0, scale=sigma, size=window.shape)
        return window + noise

    # Step 1: Zip the two arrays
    zipped_data = list(zip(variable_data, label_data))

    # To hold augmented data
    augmented_variable_data = []
    augmented_label_data = []
    labels_count = {"label_0": 0, "label_1": 0, "label_2":0, "label_3": 0, "label_4": 0, "label_5": 0, "label_6": 0}

    # Step 2, 3, 4, 5: Iterate through each window and perform operations
    for var_window, label_window in zipped_data:
        # Check if label in window is unique
        unique_labels = np.unique(label_window)
        if len(unique_labels) == 1 and unique_labels[0] != 2:
            label = unique_labels[0]  # Unique label
            #augmented_variable_data.append(var_window)
            #augmented_label_data.append(label_window)
            
            # Define number of synthetic windows based on label
            if label == 1 and labels_count["label_1"] > 1577:
                continue
            elif label == 0 and labels_count["label_0"] > 1632:
                continue
            elif label == 3 and labels_count["label_3"] > 1503:
                continue
            elif label == 4 and labels_count["label_4"] > 1581:
                continue
            
                continue
            
            if label == 1:
                labels_count["label_1"] += 11
                synthetic_count = 11
            elif label == 0:
                labels_count["label_0"] += 16
                synthetic_count = 16
            elif label == 3:
                labels_count["label_3"] += 8
                synthetic_count = 8
            elif label == 4:
                labels_count["label_4"] += 11
                synthetic_count = 11               
            else:
                synthetic_count = 0
            
            # Generate synthetic windows using JITTERING
            for _ in range(synthetic_count):
                synthetic_window = jitter(var_window)  # Use the new jittering function
                augmented_variable_data.append(synthetic_window)
                augmented_label_data.append(label_window)  # Use the same label window

    # Step 6: Unzip the augmented arrays
    augmented_variable_data = np.array(augmented_variable_data)
    augmented_label_data = np.array(augmented_label_data)

    return augmented_variable_data, augmented_label_data


In [ ]:
import numpy as np

def augment_data_with_rotation_and_translation(variable_data, label_data): 
    def rotate_and_translate(window, rotation_angle_range=(-10, 10), translation_range=(-0.1, 0.1)):
        """
        Applies random rotation and translation to the input window.
        :param window: The input data window (numpy array) with shape (timesteps, features).
        :param rotation_angle_range: Range of angles (in degrees) for random rotation.
        :param translation_range: Range for random translation for each feature.
        :return: Transformed window.
        """
        # Generate random rotation angle (converted to radians)
        rotation_angle = np.radians(np.random.uniform(*rotation_angle_range))
        cos_theta, sin_theta = np.cos(rotation_angle), np.sin(rotation_angle)
        
        # Rotation matrix for 2D data (assuming first two features are X and Y)
        rotation_matrix = np.array([[cos_theta, -sin_theta],
                                     [sin_theta,  cos_theta]])
        
        # Apply rotation (only to the first two features, if applicable)
        rotated_window = window.copy()
        if window.shape[1] >= 2:  # Apply only if there are at least two features
            rotated_window[:, :2] = np.dot(window[:, :2], rotation_matrix.T)
        
        # Apply translation to all features 
        # Ensure we don't add any extra features by using the original window's shape
        translation = np.random.uniform(*translation_range, size=window.shape[1])
        translated_window = rotated_window + translation
        
        # Ensure the returned window has exactly the same number of features as the input
        return translated_window[:, :window.shape[1]]

    # Step 1: Zip the two arrays
    zipped_data = list(zip(variable_data, label_data))

    # To hold augmented data
    augmented_variable_data = []
    augmented_label_data = []
    labels_count = {"label_0": 0, "label_1": 0, "label_2":0, "label_3": 0, "label_4": 0, "label_5": 0, "label_6": 0}
    

    # Step 2, 3, 4, 5: Iterate through each window and perform operations
    for var_window, label_window in zipped_data:
        # Check if label in window is unique
        unique_labels = np.unique(label_window)
        if len(unique_labels) == 1 and unique_labels[0] != 2:
            label = unique_labels[0]  # Unique label
            #augmented_variable_data.append(var_window)
            #augmented_label_data.append(label_window)
            
            # Define number of synthetic windows based on label
            if label == 1 and labels_count["label_1"] > 1577:
                continue
            elif label == 0 and labels_count["label_0"] > 1632:
                continue
            elif label == 3 and labels_count["label_3"] > 1503:
                continue
            elif label == 4 and labels_count["label_4"] > 1581:
                continue
            
                continue
            
            if label == 1:
                labels_count["label_1"] += 11
                synthetic_count = 11
            elif label == 0:
                labels_count["label_0"] += 16
                synthetic_count = 16
            elif label == 3:
                labels_count["label_3"] += 8
                synthetic_count = 8
            elif label == 4:
                labels_count["label_4"] += 11
                synthetic_count = 11               
            else:
                synthetic_count = 0
            
            # Generate synthetic windows using ROTATION and TRANSLATION
            for _ in range(synthetic_count):
                synthetic_window = rotate_and_translate(var_window)  # Apply rotation and translation
                augmented_variable_data.append(synthetic_window)
                augmented_label_data.append(label_window) 

    # Step 6: Unzip the augmented arrays
    # Explicitly ensure we have the correct shape
    augmented_variable_data = np.array(augmented_variable_data)
    augmented_label_data = np.array(augmented_label_data)

    return augmented_variable_data, augmented_label_data

In [ ]:
transformed_adl_data_train.label.value_counts().sort_index()

In [ ]:
#print(occurrence_transitions_slide)

In [ ]:
#X_train_data_augmentation = transformed_adl_data_train[transformed_adl_data_train.label.isin([0,1,3,4])]

In [ ]:
#X_train_data_no_augmentation = transformed_adl_data_train[~transformed_adl_data_train.label.isin([0,1,3,4])]

In [ ]:
#X_train_data_augmentation.label.value_counts()

In [ ]:
#X_train_data_no_augmentation.label.value_counts()

In [ ]:
#X_train_data_transitions.atividade.value_counts()

In [ ]:
import numpy as np

def ocurrence_unique_values(targets_test):

    # Example of 3D array with the format (256, 256, 1)
    #data = np.random.randint(0, 3, (256, 250, 1))

    # Reshape to (256, 256) to simplify the processing
    data_reshaped = targets_test.reshape(targets_test.shape[0], 151)

    # Get unique values per window (each window has 256 rows)
    # Since the second dimension always contains the same value, we can just select the first column for each window
    unique_values = data_reshaped[:, 0]

    # Count occurrences of each unique value (0, 1, 2)
    unique, counts = np.unique(unique_values, return_counts=True)

    # Create a dictionary to show the counts of 0, 1, and 2
    occurrences = dict(zip(unique, counts))
    return occurrences


In [ ]:
#segments_train_transitions,targets_train_transitions = apply_segmentation_with_sliding_window(X_train_data_transitions)

In [ ]:
#occurrence_transitions= ocurrence_unique_values(targets_train_transitions)

In [ ]:
#print(occurrence_transitions)

In [ ]:
#targets_train_transitions.shape

In [ ]:
#segments_train_transitions_no_sliding,targets_train_transitions_no_sliding = apply_segmentation_no_sliding_window(X_train_data_transitions)

In [ ]:
#occurrence_transitions_slide= ocurrence_unique_values(targets_train_transitions_no_sliding)

In [ ]:
#print(occurrence_transitions_slide)

In [ ]:
#targets_train_transitions_no_sliding.shape

In [ ]:
#X_train_data_atomic = X_train_data[X_train_data.atividade.isin([0,1,2])]

In [ ]:
#segments_train_atomic,targets_train_atomic = apply_segmentation_no_sliding_window(X_train_data_atomic)

In [ ]:
#print(segments_train_atomic.shape)

In [ ]:
#print(targets_train_atomic.shape)

In [ ]:
#segments_train_1 = np.vstack((segments_train_atomic,segments_train_transitions))

In [ ]:
#targets_train_1 = np.vstack((targets_train_atomic,targets_train_transitions))

In [ ]:
#print(segments_train_1.shape)

In [ ]:
#print(targets_train_1.shape)

In [ ]:
#count_int_classes_of_array(targets_train_1)

In [ ]:
#segments_train_atomic, targets_train_atomic = delete_segments_not_fully_filled(segments_train_atomic,targets_train_atomic)

In [ ]:
#count_int_classes_of_array(targets_train_1)

In [ ]:
#segments_train_2,targets_train_2 = augment_data_with_jittering(segments_train_atomic,targets_train_atomic)

In [ ]:
#count_int_classes_of_array(targets_train_2)

In [ ]:
segments_train_jittering,targets_train_jittering = augment_data_with_jittering(segments_train_no_sliding_window,targets_train_no_sliding_window)

In [ ]:
count_int_classes_of_array(targets_train_jittering)

In [ ]:
occurrence_crop = ocurrence_unique_values(targets_train_jittering)

In [ ]:
print(occurrence_crop)

In [ ]:
print(segments_train_jittering.shape)

In [ ]:
print(targets_train_jittering.shape)

In [ ]:
segments_train_rotation,targets_train_rotation = augment_data_with_rotation_and_translation(segments_train_no_sliding_window,targets_train_no_sliding_window)

In [ ]:
segments_train_rotation,targets_train_rotation = augment_data_with_rotation_and_translation(segments_train_no_sliding_window,targets_train_no_sliding_window)

In [ ]:
count_int_classes_of_array(targets_train_rotation)

In [ ]:
occurrence_crop = ocurrence_unique_values(targets_train_rotation)

In [ ]:
print(occurrence_crop)

In [ ]:
print(segments_train_rotation)

In [ ]:
#segments_train_rotation = segments_train_rotation[:, :, :-1]

In [ ]:
print(segments_train_rotation)

In [ ]:
print(targets_train_rotation.shape)

In [ ]:
segments_train_jittering.shape

In [ ]:
segments_train_rotation.shape

In [ ]:
segments_train_1 = np.vstack((segments_train_jittering,segments_train_rotation))

In [ ]:
targets_train_1 = np.vstack((targets_train_jittering,targets_train_rotation))

In [ ]:
print(segments_train_1.shape)

In [ ]:
print(targets_train_1.shape)

In [ ]:
print(segments_train_1.shape)
print(segments_train_1.shape)

In [ ]:
#segments_train_2,targets_train_2 = augment_data_with_time_warping(segments_train_atomic,targets_train_atomic)

In [ ]:
segments_train_warping,targets_train_warping = augment_data_with_time_warping(segments_train_no_sliding_window,targets_train_no_sliding_window)

In [ ]:
count_int_classes_of_array(targets_train_warping)

In [ ]:
occurrence_crop = ocurrence_unique_values(targets_train_warping)

In [ ]:
print(occurrence_crop)

In [ ]:
print(segments_train_warping.shape)

In [ ]:
print(targets_train_warping.shape)

In [ ]:
print(segments_train_warping)

In [ ]:
segments_train_2 = np.vstack((segments_train_1,segments_train_warping))

In [ ]:
targets_train_2 = np.vstack((targets_train_1,targets_train_warping))

In [ ]:
print(segments_train_2.shape)

In [ ]:
print(targets_train_2.shape)

In [ ]:
print(segments_train_2.shape)
print(segments_train_2.shape)

In [ ]:
#segments_train_10,targets_train_10 = augment_data_with_magnitude_scaling(segments_train_atomic,targets_train_atomic)

In [ ]:
segments_train_mag,targets_train_mag = augment_data_with_magnitude_scaling(segments_train_no_sliding_window,targets_train_no_sliding_window)

In [ ]:
count_int_classes_of_array(targets_train_mag)

In [ ]:
occurrence_crop = ocurrence_unique_values(targets_train_mag)

In [ ]:
print(occurrence_crop)

In [ ]:
print(segments_train_mag.shape)

In [ ]:
print(targets_train_mag.shape)

In [ ]:
print(segments_train_mag)

In [ ]:
segments_train_3 = np.vstack((segments_train_2,segments_train_mag))

In [ ]:
targets_train_3 = np.vstack((targets_train_2,targets_train_mag))

In [ ]:
print(segments_train_3.shape)

In [ ]:
print(targets_train_3.shape)

In [ ]:
print(segments_train_3.shape)
print(targets_train_3.shape)

In [ ]:
segments_train.shape

In [ ]:
segments_train_4 = np.vstack((segments_train_3,segments_train))

In [ ]:
targets_train_4 = np.vstack((targets_train_3,targets_train))

In [ ]:
#segments_train_5 = np.vstack((segments_train_4,segments_train_transitions))

In [ ]:
#targets_train_5 = np.vstack((targets_train_4,targets_train_transitions))

In [ ]:
segments_train = segments_train_4
targets_train = targets_train_4

In [ ]:
# Example arrays (replace these with your actual data)
# segments_train = np.random.random((n, 200, 9))
# targets_train = np.random.random((n, 200, 1))

# Find indices where the label in the last dimension of targets_train is -1.0
indices_to_keep = ~np.any(targets_train.squeeze(-1) == -1.0, axis=1)

# Filter segments_train and targets_train to keep only the rows where labels are not -1.0
segments_train_filtered = segments_train[indices_to_keep]
targets_train_filtered = targets_train[indices_to_keep]

# Print the shapes of the filtered arrays to verify
print(f"Original segments_train shape: {segments_train.shape}")
print(f"Filtered segments_train shape: {segments_train_filtered.shape}")
print(f"Original targets_train shape: {targets_train.shape}")
print(f"Filtered targets_train shape: {targets_train_filtered.shape}")

In [ ]:
count_int_classes_of_array(targets_train)

In [ ]:
# Example arrays (replace these with your actual data)
# segments_train = np.random.random((n, 200, 9))
# targets_train = np.random.random((n, 200, 1))

# Find indices where the label in the last dimension of targets_train is -1.0
indices_to_keep = ~np.any(targets_train.squeeze(-1) == -1.0, axis=1)

# Filter segments_train and targets_train to keep only the rows where labels are not -1.0
segments_train_filtered = segments_train[indices_to_keep]
targets_train_filtered = targets_train[indices_to_keep]

# Print the shapes of the filtered arrays to verify
print(f"Original segments_train shape: {segments_train.shape}")
print(f"Filtered segments_train shape: {segments_train_filtered.shape}")
print(f"Original targets_train shape: {targets_train.shape}")
print(f"Filtered targets_train shape: {targets_train_filtered.shape}")

In [ ]:
print(segments_train.shape)
print(targets_train.shape)

In [ ]:
count_int_classes_of_array(targets_train_filtered)

In [ ]:
segments_train = segments_train_filtered

In [ ]:
targets_train = targets_train_filtered

In [ ]:
targets_train = targets_train.reshape(targets_train.shape[0],151)

In [ ]:
print(targets_train.shape)

In [ ]:
labels_train = targets_train[:,0]

In [ ]:
labels_train_categorical = to_categorical(labels_train,num_classes=5)

In [ ]:
print(labels_train_categorical)

In [ ]:
category_count_categorical(labels_train_categorical)

In [ ]:
X_train = segments_train
y_train = labels_train_categorical
X_validation = segments_val
y_validation = labels_val_categorical

In [ ]:
print(X_train.shape)
print(y_train.shape)
print(X_validation.shape)
print(y_validation.shape)

In [ ]:
category_count_categorical(y_train)

In [ ]:
def shuffle_data(variable_data,label_data):
    perm_indices = np.random.permutation(variable_data.shape[0])
    # Shuffle both arrays using the same permutation indices
    shuffled_variable_data = variable_data[perm_indices]
    shuffled_label_data = label_data[perm_indices]
    return (shuffled_variable_data,shuffled_label_data)

In [ ]:
#X_train,y_train = shuffle_data(X_train,y_train)
#X_validation,y_validation = shuffle_data(X_validation,y_validation)

In [ ]:
print(X_train.shape)
print(y_train.shape)
print(X_validation.shape)
print(y_validation.shape)

In [ ]:
X_train = np.array(X_train, dtype=np.float64)
X_validation = np.array(X_validation, dtype=np.float64)

In [ ]:
#from tensorflow.keras import regularizers

In [ ]:
""" from keras.models import Model
from keras.layers import Input, LSTM, Dense, Dropout, Attention, Conv1D, MaxPooling1D, BatchNormalization
from keras.regularizers import l2
import tensorflow as tf

def lstm_attention_sensor():
    inputs = Input(shape=(151, 3))  # Input shape adjusted for accelerometer and gyroscope data
    
    # CNN Layer: Extract local features
    conv_out = Conv1D(filters=96,  # Best parameter for conv_filters
                      kernel_size=5,  # Best parameter for kernel_size
                      activation='relu')(inputs)
    conv_out = MaxPooling1D(pool_size=2)(conv_out)
    conv_out = BatchNormalization()(conv_out)  # Normalize after convolution
    
    # First LSTM layer
    lstm_out = LSTM(96,  # Best parameter for input_units_lstm1
                    kernel_regularizer=l2(0.0014023399602325875),  # Best parameter for l2_reg1
                    recurrent_regularizer=l2(0.0014023399602325875),  # Same as kernel regularizer
                    return_sequences=True)(conv_out)
    
    # Second LSTM layer
    lstm_out_2 = LSTM(96,  # Best parameter for input_units_lstm2
                      kernel_regularizer=l2(0.052415156927544435),  # Best parameter for l2_reg2
                      recurrent_regularizer=l2(0.052415156927544435),  # Same as kernel regularizer
                      return_sequences=True)(lstm_out)
    
    # Dense layer for feature extraction
    dense_out = Dense(256,  # Best parameter for dense_units
                      activation='relu')(lstm_out_2)
    dropout_out = Dropout(0.2)(dense_out)  # Best parameter for dropout_rate
    
    # Attention mechanism
    attention = Attention()([dropout_out, dropout_out])
    context_vector = tf.reduce_sum(attention, axis=1)
    
    # Output layer
    output = Dense(5, activation='softmax')(context_vector)
    
    # Model definition
    model = Model(inputs=inputs, outputs=output)
    model.compile(loss='categorical_crossentropy', 
                  optimizer=tf.keras.optimizers.Adam(learning_rate=0.003964886305018653),  # Best parameter for learning_rate
                  metrics=['accuracy'])
    
    return model
 """

In [ ]:
from keras.models import Model
from keras.layers import Input, LSTM, Dense, Dropout, Attention, Conv1D, MaxPooling1D, BatchNormalization
from keras.regularizers import l2
import tensorflow as tf

def lstm_attention_sensor(hp):
    inputs = Input(shape=(151, 3))  # Input shape adjusted for accelerometer and gyroscope data
    
    # CNN Layer: Extract local features
    conv_out = Conv1D(filters=hp.Int('conv_filters', min_value=16, max_value=256, step=16), 
                      kernel_size=hp.Choice('kernel_size', [3, 5]), 
                      activation='relu')(inputs)
    conv_out = MaxPooling1D(pool_size=2)(conv_out)
    conv_out = BatchNormalization()(conv_out)  # Normalize after convolution
    
    # First LSTM layer
    lstm_out = LSTM(hp.Int("input_units_lstm1", min_value=32, max_value=256, step=32), 
                    kernel_regularizer=l2(hp.Float('l2_reg1', min_value=0.001, max_value=0.1, sampling='log')),
                    recurrent_regularizer=l2(hp.Float('l2_reg1', min_value=0.001, max_value=0.1, sampling='log')),
                    return_sequences=True)(conv_out)
    
    # Second LSTM layer
    lstm_out_2 = LSTM(hp.Int("input_units_lstm2", min_value=32, max_value=256, step=32), 
                      kernel_regularizer=l2(hp.Float('l2_reg2', min_value=0.001, max_value=0.1, sampling='log')),
                      recurrent_regularizer=l2(hp.Float('l2_reg2', min_value=0.001, max_value=0.1, sampling='log')),
                      return_sequences=True)(lstm_out)
    
    # Dense layer for feature extraction
    dense_out = Dense(hp.Int("dense_units", min_value=32, max_value=256, step=32), activation='relu')(lstm_out_2)
    dropout_out = Dropout(hp.Float('dropout_rate', min_value=0.1, max_value=0.5, step=0.1))(dense_out)
    
    # Attention mechanism
    attention = Attention()([dropout_out, dropout_out])
    context_vector = tf.reduce_sum(attention, axis=1)
    
    # Output layer
    output = Dense(5, activation='softmax')(context_vector)
    
    # Model definition
    model = Model(inputs=inputs, outputs=output)
    model.compile(loss='categorical_crossentropy', 
                  optimizer=tf.keras.optimizers.Adam(hp.Float('learning_rate', 
                                                              min_value=1e-4, 
                                                              max_value=1e-2, 
                                                              sampling='LOG', 
                                                              default=1e-3)),
                  metrics=['accuracy'])
    
    return model


In [ ]:
#COMENTADO EM 22/11/204
""" from keras.models import Model
from keras.layers import Input, LSTM, Dense, Dropout, Attention
from keras.regularizers import l2
import tensorflow as tf

def lstm_attention_sensor(hp):
    inputs = Input(shape=(256, 9))
    
    # LSTM layer
    lstm_out = LSTM(hp.Int("input_units_lstm1", min_value=32, max_value=256, step=32), 
                    kernel_regularizer=l2(hp.Float('l2_reg1', min_value=0.001, max_value=0.1, sampling='log')),
                    recurrent_regularizer=l2(hp.Float('l2_reg1', min_value=0.001, max_value=0.1, sampling='log')),
                    return_sequences=True)(inputs)
    
    
    dense_out = Dense(hp.Int("input_units_lstm1", min_value=32, max_value=256, step=32), activation='relu')(lstm_out)
    # Attention mechanism using Keras Attention layer
    attention = Attention()([dense_out, dense_out])
    
    # Context vector computation
    context_vector = tf.reduce_sum(attention, axis=1)
    
    # Output layer
    output = Dense(6, activation='softmax')(context_vector)
    
    # Model definition
    model = Model(inputs=inputs, outputs=output)
    model.compile(loss='categorical_crossentropy', 
                  optimizer=tf.keras.optimizers.Adam(hp.Float('learning_rate', 
                                                              min_value=1e-4, 
                                                              max_value=1e-2, 
                                                              sampling='LOG', 
                                                              default=1e-3)),
                  metrics=['accuracy'])
    
    return model """


In [ ]:

""" from keras.models import Model
from keras.layers import Input, LSTM, Dense, Dropout, Activation, Permute, Multiply, Lambda, Reshape
import keras.backend as K

def lstm_attention_sensor(hp):
    inputs = Input(shape=(256,6))
    lstm_out = LSTM(hp.Int("input_units_lstm1",min_value=32,max_value=256,step=32), kernel_regularizer=l2(hp.Float('l2_reg1', min_value=0.001, max_value=0.1, sampling='log')),
             recurrent_regularizer=l2(hp.Float('l2_reg1', min_value=0.001, max_value=0.1, sampling='log')),return_sequences=True)(inputs)
    
    # Attention mechanism
    attention = Dense(hp.Int("input_units_lstm1",min_value=32,max_value=256,step=32), activation='relu')(lstm_out)    
    attention = Permute([2, 1])(attention)
    attention = Activation('softmax')(attention)
    attention = Permute([2, 1])(attention)
    
    context_vector = Multiply()([lstm_out, attention])
    context_vector = Lambda(lambda x: K.sum(x, axis=1))(context_vector)    
    output = Dense(6, activation='softmax')(context_vector)
    
    model = Model(inputs=inputs, outputs=output)
    model.compile(loss='categorical_crossentropy', optimizer=tf.keras.optimizers.Adam(hp.Float(
                    'learning_rate',
                    min_value=1e-4,
                    max_value=1e-2,
                    sampling='LOG',
                    default=1e-3
                )), metrics=['accuracy'])
    return model
 """

In [ ]:
tuner = RandomSearch(
    lstm_attention_sensor,
    objective = 'val_accuracy', 
    seed=42,
    max_trials = 5,
    overwrite=True,        
    executions_per_trial = 1,    
    directory = LOG_DIR,
    project_name="lstm_acel_giro_multiclass")

In [ ]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    min_delta = 0,
    verbose=1,
    restore_best_weights=True)

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=1,
    verbose=1)

tensorboard = tf.keras.callbacks.TensorBoard(LOG_DIR)

my_callbacks = [early_stopping,reduce_lr,tensorboard]  


In [ ]:
from sklearn.utils.class_weight import compute_class_weight

# Assuming y_train is one-hot encoded, convert it to integer labels
y_train_labels = np.argmax(y_train, axis=1)  # Converts one-hot to class indices

# Compute class weights based on the integer labels
class_weights = compute_class_weight('balanced', classes=np.unique(y_train_labels), y=y_train_labels)

# Convert the class weights into a dictionary
class_weight_dict = {i: weight for i, weight in enumerate(class_weights)}

print(class_weight_dict.values())

In [ ]:
""" model = lstm_attention_sensor()
model.fit(x=X_train,
            y=y_train,
            epochs=50,
            validation_data=(X_validation,y_validation), callbacks=my_callbacks,batch_size=64, use_multiprocessing=True) """

In [ ]:
tuner.search(x=X_train,
            y=y_train,
            epochs=50,
            validation_data=(X_validation,y_validation), callbacks=my_callbacks,batch_size=64, use_multiprocessing=True)

In [ ]:
tuner.results_summary()

In [ ]:
tuner.get_best_hyperparameters()[0].values

In [ ]:
best_model = tuner.get_best_models(num_models=1)[0]

In [ ]:
best_model.history

In [ ]:
best_model.summary()

In [ ]:
loss, accuracy = best_model.evaluate(X_validation, y_validation)

In [ ]:
predictions = best_model.predict(X_validation)

In [ ]:
print(y_validation)

In [ ]:
print(y_validation.shape)

In [ ]:
y_validation_inverse = np.argmax(y_validation, axis=1)

In [ ]:
print(y_validation_inverse)

In [ ]:
print(len(y_validation_inverse))

In [ ]:
print(predictions)

In [ ]:
classes_x=np.argmax(predictions,axis=1)

In [ ]:
print(classes_x)

In [ ]:
print(len(classes_x))

In [ ]:
print(classes_x.shape)

In [ ]:
from sklearn.metrics import confusion_matrix,ConfusionMatrixDisplay,classification_report,accuracy_score
cm = confusion_matrix(y_validation_inverse, classes_x)

In [ ]:
disp = ConfusionMatrixDisplay(confusion_matrix=cm)

In [ ]:
disp.plot()

In [ ]:
print(classification_report(y_validation_inverse, classes_x))

In [ ]:
from sklearn.metrics import confusion_matrix,ConfusionMatrixDisplay,classification_report,accuracy_score
# Calculate accuracy for each class
class_accuracies = {}
for cls in np.unique(classes_x):
    class_mask = (y_validation_inverse == cls)
    class_accuracies[f"Class {cls}"] = accuracy_score(y_validation_inverse[class_mask], classes_x[class_mask])

# Print class-wise accuracies
print("Accuracy for each class:")
for cls, acc in class_accuracies.items():
    print(f"{cls}: {acc:.2f}")

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

# Example function to calculate per-class accuracy
def calculate_per_class_accuracy(y_true, y_pred, class_names=None):
    """
    Calculate accuracy for each class.
    
    Parameters:
    - y_true: Ground truth labels (list or numpy array).
    - y_pred: Predicted labels (list or numpy array).
    - class_names: List of class names (optional).
    
    Returns:
    - A dictionary with class-wise accuracies.
    """
    # Generate confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    
    # Calculate accuracy for each class
    per_class_accuracy = cm.diagonal() / cm.sum(axis=1)
    
    # Organize results
    results = {}
    if class_names is None:
        class_names = [f'Class {i}' for i in range(len(per_class_accuracy))]
    for class_name, accuracy in zip(class_names, per_class_accuracy):
        results[class_name] = accuracy
    
    return results

# Example usage
if __name__ == "__main__":
    # Example true and predicted labels
    #y_true = [0, 1, 2, 2, 0, 1, 1, 0, 2, 1]  # Ground truth
    #y_pred = [0, 1, 1, 2, 0, 1, 2, 0, 2, 1]  # Predictions
    
    # Class names (optional)
    class_names = [0, 1, 2,3,4,5,6]
    
    # Calculate and display per-class accuracy
    accuracy_per_class = calculate_per_class_accuracy(y_validation_inverse, classes_x, class_names)
    for class_name, accuracy in accuracy_per_class.items():
        print(f"{class_name}: {accuracy:.2f}")
    
    # Display class


In [ ]:
print(classification_report(y_validation_inverse, classes_x))

In [ ]:
from sklearn.metrics import confusion_matrix,ConfusionMatrixDisplay,classification_report,accuracy_score
# Calculate accuracy for each class
class_accuracies = {}
for cls in np.unique(classes_x):
    class_mask = (y_validation_inverse == cls)
    class_accuracies[f"Class {cls}"] = accuracy_score(y_validation_inverse[class_mask], classes_x[class_mask])

# Print class-wise accuracies
print("Accuracy for each class:")
for cls, acc in class_accuracies.items():
    print(f"{cls}: {acc:.2f}")

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

# Example function to calculate per-class accuracy
def calculate_per_class_accuracy(y_true, y_pred, class_names=None):
    """
    Calculate accuracy for each class.
    
    Parameters:
    - y_true: Ground truth labels (list or numpy array).
    - y_pred: Predicted labels (list or numpy array).
    - class_names: List of class names (optional).
    
    Returns:
    - A dictionary with class-wise accuracies.
    """
    # Generate confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    
    # Calculate accuracy for each class
    per_class_accuracy = cm.diagonal() / cm.sum(axis=1)
    
    # Organize results
    results = {}
    if class_names is None:
        class_names = [f'Class {i}' for i in range(len(per_class_accuracy))]
    for class_name, accuracy in zip(class_names, per_class_accuracy):
        results[class_name] = accuracy
    
    return results

# Example usage
if __name__ == "__main__":
    # Example true and predicted labels
    #y_true = [0, 1, 2, 2, 0, 1, 1, 0, 2, 1]  # Ground truth
    #y_pred = [0, 1, 1, 2, 0, 1, 2, 0, 2, 1]  # Predictions
    
    # Class names (optional)
    class_names = [0, 1, 2,3,4,5,6]
    
    # Calculate and display per-class accuracy
    accuracy_per_class = calculate_per_class_accuracy(y_validation_inverse, classes_x, class_names)
    for class_name, accuracy in accuracy_per_class.items():
        print(f"{class_name}: {accuracy:.2f}")
    
    # Display class


In [ ]:
best_model.save(WORKDIR+"/Model/best_models_saved_"+data_hora)